# ADME ACZ Silver Layer

Use this notebook to transform OSDU records from an Azure Data Manager for Energy Analytics Consumption Zone (ACZ) into Silver Layer Delta tables.

Before you run it, confirm that the customer has:

- A running Azure Data Manager for Energy instance.
- A configured Analytics Consumption Zone.
- A Microsoft Fabric lakehouse with access to the ACZ bronze Delta table.

For the full overview, prerequisites, configuration reference, and operating guidance, see `README.md`.


## Architecture

The notebook executes the Silver Layer transformation pipeline in Microsoft Fabric:

1. Read OSDU records from the ACZ bronze Delta table.
2. Resolve OSDU kind schemas from the public schema registry.
3. Infer Spark types and flatten scalar and object fields.
4. Create parent and child Delta tables, or create one reassembled table per kind.
5. Write Silver Layer Delta tables for downstream analytics, reporting, and data engineering workloads.

The README owns the durable architecture and operating guidance. This notebook keeps the executable steps close to the code cells that run them.


## Spark runtime configuration

Prepare Spark before loading pipeline logic. The next cell reuses the active Fabric Spark session when available, creates one outside Fabric when needed, and applies safe defaults for Delta writes and decomposition workloads.


In [ ]:
import logging
import os

from pyspark.sql import SparkSession

# Configure logging at the beginning
logging.basicConfig(
    level=logging.WARN,
    format='%(asctime)s - %(levelname)s - %(filename)s:%(lineno)d - %(message)s'
)

# Fabric provides the active SparkSession in this notebook.
if "spark" not in globals() or spark is None:
    raise RuntimeError(
        "This notebook expects the Fabric-provided spark session. Attach the notebook to a Fabric runtime and rerun."
    )


def _env_or_default(name: str, default: str) -> str:
    val = os.environ.get(name)
    return val if val not in (None, "") else default


SPARK_CONFIG_DEFAULTS = {
    # Core SQL settings
    "spark.sql.session.timeZone": _env_or_default("ADME_SPARK_TIMEZONE", "UTC"),
    "spark.sql.adaptive.enabled": _env_or_default("ADME_SPARK_ADAPTIVE", "true"),
    "spark.sql.adaptive.coalescePartitions.enabled": _env_or_default("ADME_SPARK_COALESCE", "true"),
    "spark.sql.shuffle.partitions": _env_or_default("ADME_SPARK_SHUFFLE_PARTITIONS", "200"),
    "spark.sql.execution.arrow.pyspark.enabled": _env_or_default("ADME_SPARK_ARROW", "true"),
    # Delta Lake write optimizations
    "spark.microsoft.delta.optimizeWrite.enabled": _env_or_default("ADME_DELTA_OPTIMIZE_WRITE", "true"),
    "spark.microsoft.delta.optimizeWrite.binSize": _env_or_default("ADME_DELTA_BIN_SIZE", "1073741824"),  # 1GB
    "spark.databricks.delta.autoCompact.enabled": _env_or_default("ADME_DELTA_AUTO_COMPACT", "true"),
    # Fabric DirectLake compatibility (V-Order)
    "spark.sql.parquet.vorder.enabled": _env_or_default("ADME_PARQUET_VORDER", "true"),
    # File size control for better performance
    "spark.sql.files.maxPartitionBytes": _env_or_default("ADME_MAX_PARTITION_BYTES", "536870912"),  # 512MB
}

# Optional schema evolution style writes.
if _env_or_default("ADME_SPARK_AUTO_MERGE", "false").lower() == "true":
    SPARK_CONFIG_DEFAULTS["spark.databricks.delta.schema.autoMerge.enabled"] = "true"

for k, v in SPARK_CONFIG_DEFAULTS.items():
    spark.conf.set(k, v)

print("Spark session ready")
print(f"  appName: {spark.sparkContext.appName}")
print("Effective Spark config:")
for k in sorted(SPARK_CONFIG_DEFAULTS):
    try:
        print(f"  {k} = {spark.conf.get(k)}")
    except Exception:
        print(f"  {k} = <unavailable>")

## Configuration

Edit the explicit tenant configuration block in the next cell before running the notebook in a new workspace. Environment variables can still override the same values for scheduled execution.

Required decisions:

- Set `WORKSPACE_ID` and `LAKEHOUSE_ID`, or attach a lakehouse so Fabric can resolve them automatically.
- Set `BRONZE_TABLE` to the ACZ bronze Delta table that contains OSDU records.
- Select `KINDS` for the OSDU kinds to process.
- Use `RUN_PROFILE = "interactive"` to review settings, `RUN_PROFILE = "dry_run"` to validate and preview writes, or `RUN_PROFILE = "full"` to execute.
- Choose `OUTPUT_MODE = "normalized"` for parent and child tables, or `OUTPUT_MODE = "wide"` for one wide table per kind.

The cell also configures schema caching, run manifest output, and output table naming.


In [ ]:
import hashlib
import json
import os

# ════════════════════════════════════════════════════════════════════
# EXPLICIT TENANT CONFIGURATION - Edit this block for each workspace
# ════════════════════════════════════════════════════════════════════

# Leave WORKSPACE_ID and LAKEHOUSE_ID blank to use the attached Fabric lakehouse context.
WORKSPACE_ID = ""
LAKEHOUSE_ID = ""
BRONZE_TABLE = "osducatalog"
NOTEBOOK_VERSION = "0.2.0"

RUN_PROFILE = "interactive"  # "interactive" | "dry_run" | "full"

# OSDU kinds to process (edit this list directly)
KINDS = [
    "osdu:wks:work-product-component--WellLog:1.4.0",
    "osdu:wks:work-product-component--WellboreTrajectory:1.3.0",
]

INCREMENTAL = False     # True = upsert parent rows and replace child rows for changed parent ids
ALLOW_OVERWRITE = False # True = allow full refresh to replace existing output tables
LIMIT = 0               # max records per kind (0 = no limit)
OUTPUT_MODE = "normalized"  # "normalized" = parent+children | "wide" = one table per kind
DROP_WKT = True         # True = drop WKT geometry columns
TABLE_PREFIX = ""       # optional prefix for output tables

# Repeatability helpers
PERSIST_SCHEMA_CACHE = True
SCHEMA_CACHE_TABLE = "silver_schema_cache"
RUN_MANIFEST_TABLE = "silver_run_manifest"

# ════════════════════════════════════════════════════════════════════
# FABRIC RUNTIME RESOLUTION
# ════════════════════════════════════════════════════════════════════


def _spark_conf_get_optional(key: str) -> str | None:
    try:
        try:
            val = spark.conf.get(key, None)
        except TypeError:
            val = spark.conf.get(key)
        if val is None:
            return None
        sval = str(val).strip()
        return sval if sval else None
    except Exception:
        return None


def _resolve_workspace_id(explicit_value: str | None = None) -> str:
    explicit = (explicit_value or "").strip()
    if explicit:
        return explicit

    val = _spark_conf_get_optional("trident.workspace.id")
    if val:
        print(f"  spark.conf[trident.workspace.id] = {val}")
        return val
    raise ValueError(
        "Fabric workspace id could not be resolved. Attach a lakehouse, set WORKSPACE_ID, or set ADME_WORKSPACE_ID."
    )


def _resolve_lakehouse_id(explicit_value: str | None = None) -> str:
    explicit = (explicit_value or "").strip()
    if explicit:
        return explicit

    candidate_keys = [
        "trident.lakehouse.id",
        "trident.defaultLakehouse.id",
        "trident.lakehouseID",
    ]
    for key in candidate_keys:
        val = _spark_conf_get_optional(key)
        if val:
            print(f"  spark.conf[{key}] = {val}")
            return val
        print(f"  spark.conf[{key}] unavailable in this runtime")
    raise ValueError(
        "Fabric lakehouse id could not be resolved. Attach a lakehouse, set LAKEHOUSE_ID, or set ADME_LAKEHOUSE_ID."
    )


def _normalize_output_mode(value: str | None) -> str:
    normalized = (value or "normalized").strip().lower().replace("-", "_")
    aliases = {
        "normalized": "normalized",
        "normalised": "normalized",
        "parent_child": "normalized",
        "parent_children": "normalized",
        "parent+children": "normalized",
        "wide": "wide",
        "flat": "wide",
        "reassembled": "wide",
        "reassemble": "wide",
    }
    if normalized not in aliases:
        raise ValueError("OUTPUT_MODE must be 'normalized' or 'wide'.")
    return aliases[normalized]


def _env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y"}


print("=== Configuration ===")

# Resolve workspace/lakehouse (env var takes precedence over explicit config, then spark conf)
_ws_env = os.environ.get("ADME_WORKSPACE_ID")
print(f"  ADME_WORKSPACE_ID env = {_ws_env!r}")
workspace_id = _ws_env or _resolve_workspace_id(WORKSPACE_ID)

_lh_env = os.environ.get("ADME_LAKEHOUSE_ID")
print(f"  ADME_LAKEHOUSE_ID env = {_lh_env!r}")
lakehouse_id = _lh_env or _resolve_lakehouse_id(LAKEHOUSE_ID)

bronze_table = os.environ.get("ADME_BRONZE_TABLE", BRONZE_TABLE)

print(f"\n  workspace_id = {workspace_id}")
print(f"  lakehouse_id = {lakehouse_id}")
print(f"  bronze_table = {bronze_table}")

# ════════════════════════════════════════════════════════════════════
# DERIVED CONFIGURATION
# ════════════════════════════════════════════════════════════════════

# OSDU kinds to process (env var override or user control)
_env_kinds = os.environ.get("ADME_KINDS")
if _env_kinds:
    kinds = [k.strip() for k in _env_kinds.split(",") if k.strip()]
else:
    kinds = KINDS

# Run profile (env var override or user control)
run_profile = os.environ.get("ADME_RUN_PROFILE") or RUN_PROFILE
run_profile = run_profile.strip().lower()
if run_profile not in {"interactive", "dry_run", "full"}:
    run_profile = "interactive"

# Pipeline behavior (env var override or user control)
incremental = _env_bool("ADME_INCREMENTAL", INCREMENTAL)
allow_overwrite = _env_bool("ADME_ALLOW_OVERWRITE", ALLOW_OVERWRITE)
limit_str = os.environ.get("ADME_LIMIT")
limit = int(limit_str) if limit_str else (LIMIT if LIMIT else None)
if limit == 0:
    limit = None

drop_wkt = _env_bool("ADME_DROP_WKT", DROP_WKT)

_output_mode_env = os.environ.get("ADME_OUTPUT_MODE")
if _output_mode_env:
    output_mode = _normalize_output_mode(_output_mode_env)
elif os.environ.get("ADME_REASSEMBLE"):
    # Backward-compatible alias for older scheduled runs.
    output_mode = "wide" if os.environ.get("ADME_REASSEMBLE", "").lower() != "false" else "normalized"
else:
    output_mode = _normalize_output_mode(OUTPUT_MODE)
reassemble = output_mode == "wide"

table_prefix = os.environ.get("ADME_TABLE_PREFIX", TABLE_PREFIX)
persist_schema_cache = _env_bool("ADME_PERSIST_SCHEMA_CACHE", PERSIST_SCHEMA_CACHE)
schema_cache_table = os.environ.get("ADME_SCHEMA_CACHE_TABLE", SCHEMA_CACHE_TABLE)
run_manifest_table = os.environ.get("ADME_RUN_MANIFEST_TABLE", RUN_MANIFEST_TABLE)
schema_cache_writes_enabled = persist_schema_cache and run_profile == "full"

config_snapshot = {
    "notebook_version": NOTEBOOK_VERSION,
    "workspace_id": workspace_id,
    "lakehouse_id": lakehouse_id,
    "bronze_table": bronze_table,
    "kinds": kinds,
    "run_profile": run_profile,
    "incremental": incremental,
    "allow_overwrite": allow_overwrite,
    "limit": limit,
    "output_mode": output_mode,
    "drop_wkt": drop_wkt,
    "table_prefix": table_prefix,
    "persist_schema_cache": persist_schema_cache,
    "schema_cache_table": schema_cache_table,
    "run_manifest_table": run_manifest_table,
}
config_hash = hashlib.sha256(json.dumps(config_snapshot, sort_keys=True).encode("utf-8")).hexdigest()[:12]

# Schema source (web registry with optional persisted cache)
schema_source_mode = "web-cache" if persist_schema_cache else "web"

# Print effective configuration
print("\n=== Effective Configuration ===")
print(f"  Notebook version: {NOTEBOOK_VERSION}")
print(f"  Config hash: {config_hash}")
print(f"  Run profile: {run_profile}")
print(f"  Schema source: {schema_source_mode}")
print(f"  Kinds selected: {len(kinds)}")
if kinds:
    preview = kinds[:3]
    print(f"    {preview}")
    if len(kinds) > len(preview):
        print(f"    ... and {len(kinds) - len(preview)} more")
print(f"  Load mode: {'incremental' if incremental else 'full refresh'}")
print(f"  Allow overwrite: {allow_overwrite}")
print(f"  Record limit: {limit or 'none'}")
print(f"  Output mode: {output_mode}")
print(f"  Drop WKT: {drop_wkt}")
print(f"  Table prefix: {table_prefix!r}")
print(f"  Persist schema cache: {persist_schema_cache}")
print(f"  Schema cache writes enabled: {schema_cache_writes_enabled}")
print(f"  Schema cache table: {schema_cache_table}")
print(f"  Run manifest table: {run_manifest_table}")
print("=" * 32)


## Pipeline constants

Define shared endpoints, storage scopes, run metadata schema, and per-kind result types used by the pipeline.


In [ ]:
import traceback
import uuid
from dataclasses import dataclass
from datetime import UTC, datetime

from pyspark.sql import types as T

# ── External service endpoints ───────────────────────────────────────

# OSDU schema registry web endpoint
WEB_RAW_ROOT = "https://community.opengroup.org/osdu/data/data-definitions/-/raw/master/"

# Azure Storage scope for Fabric authentication
FABRIC_STORAGE_SCOPE = "https://storage.azure.com/.default"

# ── Run-info schema ──────────────────────────────────────────────────

RUN_INFO_SCHEMA = T.StructType(
    [
        T.StructField("run_id", T.StringType(), False),
        T.StructField("kind", T.StringType(), False),
        T.StructField("start_time", T.TimestampType(), False),
        T.StructField("end_time", T.TimestampType(), True),
        T.StructField("records_processed", T.LongType(), True),
        T.StructField("records_failed", T.LongType(), True),
        T.StructField("status", T.StringType(), False),
        T.StructField("error_message", T.StringType(), True),
    ]
)

SCHEMA_CACHE_SCHEMA = T.StructType(
    [
        T.StructField("kind", T.StringType(), False),
        T.StructField("source", T.StringType(), False),
        T.StructField("schema_json", T.StringType(), False),
        T.StructField("cached_at", T.TimestampType(), False),
    ]
)

RUN_MANIFEST_SCHEMA = T.StructType(
    [
        T.StructField("run_id", T.StringType(), False),
        T.StructField("kind", T.StringType(), False),
        T.StructField("output_mode", T.StringType(), False),
        T.StructField("notebook_version", T.StringType(), True),
        T.StructField("run_profile", T.StringType(), True),
        T.StructField("config_hash", T.StringType(), True),
        T.StructField("allow_overwrite", T.BooleanType(), True),
        T.StructField("table_prefix", T.StringType(), True),
        T.StructField("bronze_table", T.StringType(), True),
        T.StructField("parent_table", T.StringType(), True),
        T.StructField("child_tables", T.ArrayType(T.StringType()), True),
        T.StructField("records_processed", T.LongType(), True),
        T.StructField("status", T.StringType(), False),
        T.StructField("error_message", T.StringType(), True),
        T.StructField("created_at", T.TimestampType(), False),
    ]
)


@dataclass
class KindResult:
    """Outcome of processing one kind."""

    kind: str
    status: str
    records_processed: int = 0
    records_failed: int = 0
    parent_table: str = ""
    child_tables: list[str] | None = None
    reassembled: bool = False
    validation_passed: bool = True
    error: str | None = None

## Helper functions

Load Fabric, OneLake, Delta write, upsert, and bronze-read helpers. These helpers use the attached lakehouse catalog first and fall back to OneLake paths when a catalog table is not available.


In [ ]:
import logging
import os
from datetime import datetime
from typing import Any

import requests
from azure.identity import DefaultAzureCredential
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

logger = logging.getLogger(__name__)


def _requires_path_fallback(exc: Exception) -> bool:
    msg = str(exc)
    return (
        "No default context found" in msg
        or "partial namespaces" in msg
        or "attach a lakehouse" in msg.lower()
    )


def _table_path_uri(table_name: str) -> str:
    ws = globals().get("workspace_id")
    lh = globals().get("lakehouse_id")
    if not ws or not lh:
        raise ValueError(
            "workspace_id/lakehouse_id not resolved; run Configuration cell before pipeline execution."
        )
    if "_onelake_table_uri" not in globals():
        raise ValueError("_onelake_table_uri is unavailable; run helper cells before pipeline execution.")
    return _onelake_table_uri(ws, lh, table_name)


# Fabric-only helpers
def kind_to_table_name(kind: str) -> str:
    # osdu:wks:master-data--Well:1.2.0 -> well
    # osdu:wks:work-product-component--WellLog:1.4.0 -> welllog
    try:
        _, _, entity_ver = kind.split(":", 2)
        entity, _ = entity_ver.rsplit(":", 1)
    except ValueError:
        entity = kind
    entity_base = entity.split("--")[-1]
    return entity_base.replace("-", "_").replace(".", "_").lower()


def table_uri(workspace_id: str, lakehouse_id: str, table_name: str) -> str:
    # In Fabric with attached lakehouse, tables are resolved via catalog.
    # Without attached context we transparently fall back to OneLake path.
    return table_name


def _write_table(df: DataFrame, target: str, mode: str = "overwrite") -> None:
    writer = df.write.format("delta").mode(mode)
    if mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    try:
        writer.saveAsTable(target)
    except Exception as exc:
        if not _requires_path_fallback(exc):
            raise
        target_uri = _table_path_uri(target)
        logger.info("No default lakehouse context; writing by path: %s", target_uri)
        writer.save(target_uri)


def write_silver_table(df: DataFrame, target: str, mode: str = "overwrite") -> None:
    _write_table(df, target, mode=mode)


def upsert_silver_table(df: DataFrame, target: str, merge_key: str = "id") -> None:
    from delta.tables import DeltaTable

    try:
        exists = spark.catalog.tableExists(target)
    except Exception as exc:
        if _requires_path_fallback(exc):
            exists = False
        else:
            raise

    if exists:
        dt = DeltaTable.forName(spark, target)
        source_alias = "s"
        target_alias = "t"
        cond = f"{target_alias}.{merge_key} = {source_alias}.{merge_key}"
        (
            dt.alias(target_alias)
            .merge(df.alias(source_alias), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        return

    # Try path-based merge/upsert when no default context is attached.
    try:
        target_uri = _table_path_uri(target)
        dt_path = DeltaTable.forPath(spark, target_uri)
        source_alias = "s"
        target_alias = "t"
        cond = f"{target_alias}.{merge_key} = {source_alias}.{merge_key}"
        (
            dt_path.alias(target_alias)
            .merge(df.alias(source_alias), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        return
    except Exception:
        _write_table(df, target, mode="overwrite")


# Fabric storage helpers

_FABRIC_CREDENTIAL = DefaultAzureCredential()


def _fabric_storage_options() -> dict[str, str]:
    token = _FABRIC_CREDENTIAL.get_token(FABRIC_STORAGE_SCOPE).token
    return {
        "bearer_token": token,
        "use_fabric_endpoint": "true",
    }


def _onelake_table_uri(workspace_id: str, lakehouse_id: str, table_name: str) -> str:
    lakehouse_ref = lakehouse_id or os.environ.get("ADME_LAKEHOUSE_NAME", "osducatalog")
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_ref}/Tables/{table_name}"


def read_bronze_kind_spark(
    kind: str,
    spark: SparkSession,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str | None = None,
    limit: int | None = None,
) -> DataFrame:
    bronze_table = bronze_table or globals().get("bronze_table")
    if not bronze_table:
        raise ValueError("bronze_table is not configured")

    # Try attached lakehouse table first; fall back to OneLake path when context is missing.
    df = None
    try:
        if spark.catalog.tableExists(bronze_table):
            df = spark.table(bronze_table)
    except Exception as exc:
        if not _requires_path_fallback(exc):
            raise
        logger.info("No default lakehouse context; resolving bronze via OneLake path")

    if df is None:
        bronze_uri = _onelake_table_uri(workspace_id, lakehouse_id, bronze_table)
        logger.info("Bronze table not attached; trying OneLake path: %s", bronze_uri)
        try:
            df = spark.read.format("delta").load(bronze_uri)
        except Exception as exc:
            raise ValueError(
                f"Bronze table '{bronze_table}' was not found in the attached catalog and OneLake load failed from '{bronze_uri}'. "
                "Attach a lakehouse or ensure ADME_WORKSPACE_ID/ADME_LAKEHOUSE_ID/ADME_BRONZE_TABLE point to a valid OneLake Delta table."
            ) from exc

    if "kind" in df.columns:
        df = df.filter(F.col("kind") == F.lit(kind))

    return df.limit(limit) if limit else df

## Core decomposition and reassembly logic

Load the standalone Silver Layer transformation logic. This section resolves OSDU schemas, classifies columns, builds parent and child tables, and optionally reassembles child data into one wide table per kind.

Do not edit this section for documentation cleanup. If transformation behavior changes are needed, update and test the source implementation first, then sync this notebook copy.


In [ ]:
# ── Core standalone schema/decompose/reassemble ─────────────────────
# This cell contains the standalone Silver Layer transformation implementation:
#   decompose.py
#   reassemble.py
#   schema_registry.py  (parsing + queries only — no Spark/Delta loaders)
#   naming.py           (child_table_name only — kind_to_table_name lives in helpers cell)
#   types.py            (DecomposedKind only)
#
# Only standalone-specific change: SchemaRegistry is built from OSDU web schemas
# (load_schema_doc) instead of the osdu_schemas Delta table.
# DO NOT EDIT decomposition logic here unless you are intentionally changing transformation behavior.

from __future__ import annotations

import json
import logging
from collections import deque
from dataclasses import dataclass
from functools import reduce
from time import perf_counter
from datetime import UTC, datetime
from typing import Any

import requests
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

logger = logging.getLogger(__name__)

# ── Web schema loader ────────────────────────────────────────────────


def parse_kind(kind: str) -> dict[str, str]:
    authority, source, entity_ver = kind.split(":", 2)
    entity, version = entity_ver.rsplit(":", 1)
    return {
        "authority": authority,
        "source": source,
        "entity": entity,
        "version": version,
        "entity_base": entity.split("--")[-1],
    }


def normalize_kind_ref(ref: str | None) -> str | None:
    if not ref:
        return None
    normalized = ref.replace("{{schema-authority}}", "osdu")
    normalized = normalized.replace("{{wksNameSpace}}", "wks")
    normalized = normalized.replace("{{wksVersion}}", "1.0.0")
    return normalized


def source_folder_for_kind(source: str, entity: str) -> str:
    if source == "wks":
        if "--" in entity:
            return entity.split("--", 1)[0]
        return "work-product-component"
    return source


_SCHEMA_DOC_CACHE: dict[str, tuple[dict[str, Any], str]] = {}


def _persistent_schema_cache_enabled() -> bool:
    return bool(globals().get("persist_schema_cache", False))


def _schema_cache_table_name() -> str:
    return globals().get("schema_cache_table", "silver_schema_cache")


def _read_table_for_cache(table_name: str) -> DataFrame:
    try:
        return spark.table(table_name)
    except Exception as exc:
        if "_requires_path_fallback" not in globals() or not _requires_path_fallback(exc):
            raise
        return spark.read.format("delta").load(_table_path_uri(table_name))


def _load_schema_doc_from_persistent_cache(kind: str) -> tuple[dict[str, Any], str] | None:
    if not _persistent_schema_cache_enabled() or "spark" not in globals() or "_table_exists" not in globals():
        return None

    table_name = _schema_cache_table_name()
    if not _table_exists(spark, table_name):
        return None

    rows = (
        _read_table_for_cache(table_name)
        .filter(F.col("kind") == F.lit(kind))
        .orderBy(F.col("cached_at").desc())
        .limit(1)
        .collect()
    )
    if not rows:
        return None

    row = rows[0]
    return json.loads(row["schema_json"]), row["source"]


def _write_schema_doc_to_persistent_cache(kind: str, schema_doc: dict[str, Any], source: str) -> None:
    if not globals().get("schema_cache_writes_enabled", False) or "spark" not in globals() or "_write_table" not in globals():
        return

    table_name = _schema_cache_table_name()
    row = [(kind, source, json.dumps(schema_doc, sort_keys=True), datetime.now(UTC))]
    df = spark.createDataFrame(row, schema=SCHEMA_CACHE_SCHEMA)
    mode = "append" if "_table_exists" in globals() and _table_exists(spark, table_name) else "overwrite"
    _write_table(df, table_name, mode=mode)


def load_schema_doc(kind: str, timeout: int = 30) -> tuple[dict[str, Any], str]:
    if kind in _SCHEMA_DOC_CACHE:
        logger.info("Using cached schema for %s", kind)
        return _SCHEMA_DOC_CACHE[kind]

    cached = _load_schema_doc_from_persistent_cache(kind)
    if cached is not None:
        logger.info("Using persisted schema cache for %s", kind)
        _SCHEMA_DOC_CACHE[kind] = cached
        return cached

    p = parse_kind(kind)
    folder = source_folder_for_kind(p["source"], p["entity"])
    candidates = [p["entity_base"]]
    if p["entity"] != p["entity_base"]:
        candidates.append(p["entity"])

    tried_urls: list[str] = []
    for name in dict.fromkeys(candidates):
        web_url = (
            f"{WEB_RAW_ROOT}SchemaRegistrationResources/shared-schemas/"
            f"{p['authority']}/{folder}/{name}.{p['version']}.json"
        )
        tried_urls.append(web_url)
        resp = requests.get(web_url, timeout=timeout)
        if resp.ok:
            schema_doc = resp.json()
            source = f"web:{web_url}"
            schema_result = (schema_doc, source)
            _SCHEMA_DOC_CACHE[kind] = schema_result
            _write_schema_doc_to_persistent_cache(kind, schema_doc, source)
            return schema_result

    raise requests.HTTPError(
        f"Schema not found for {kind}. Tried: {' | '.join(tried_urls)}"
    )


# ── naming.child_table_name ───────────────────────────────────────────


def child_table_name(parent: str, array_field: str) -> str:
    """Build child table name using triple-underscore separator."""
    return f"{parent}___{array_field}"


# ── types.DecomposedKind ──────────────────────────────────────────────


@dataclass
class DecomposedKind:
    """Result of decomposing one OSDU kind into parent + children."""

    kind_name: str
    col_classes: dict[str, list[str]]
    parent: Any  # DataFrame
    children: dict[str, tuple[str, Any]]  # table_name -> (source_col, df)


# ── schema_registry.py ────────────────────────────────────────────────
#    minus from_spark / from_delta / from_dataframe (Delta-table loaders unused here) ──

_JSON_TYPE_MAP: dict[str, T.DataType] = {
    "string": T.StringType(),
    "integer": T.LongType(),
    "number": T.DoubleType(),
    "boolean": T.BooleanType(),
}


def _resolve_node(
    node: dict[str, Any],
    definitions: dict[str, Any] | None,
) -> dict[str, Any]:
    if not isinstance(node, dict):
        return node

    if "$ref" in node and definitions:
        ref = node["$ref"]
        if ref.startswith("#/definitions/"):
            resolved = definitions.get(ref[len("#/definitions/") :])
            if resolved:
                return _resolve_node(resolved, definitions)

    if "allOf" in node:
        merged: dict[str, Any] = {}
        for sub in node["allOf"]:
            if not isinstance(sub, dict):
                continue
            resolved = _resolve_node(sub, definitions)
            if "properties" in resolved:
                merged.update(resolved["properties"])
        if merged:
            return {"type": "object", "properties": merged}

    return node


def _json_schema_to_spark(
    schema_node: dict[str, Any],
    definitions: dict[str, Any] | None = None,
) -> T.DataType:
    if not isinstance(schema_node, dict):
        return T.StringType()

    json_type = schema_node.get("type")

    if "$ref" in schema_node and not json_type:
        if definitions:
            ref = schema_node["$ref"]
            if ref.startswith("#/definitions/"):
                def_key = ref[len("#/definitions/") :]
                resolved = definitions.get(def_key)
                if resolved:
                    return _json_schema_to_spark(resolved, definitions)
        return T.StringType()

    if "allOf" in schema_node and not json_type:
        merged_props: dict[str, Any] = {}
        for sub in schema_node["allOf"]:
            if not isinstance(sub, dict):
                continue
            resolved = sub
            if "$ref" in sub and definitions:
                ref = sub["$ref"]
                if ref.startswith("#/definitions/"):
                    r = definitions.get(ref[len("#/definitions/") :])
                    if r:
                        resolved = r
            if "properties" in resolved:
                merged_props.update(resolved["properties"])
        if merged_props:
            fields = []
            for name, prop_schema in merged_props.items():
                spark_type = _json_schema_to_spark(prop_schema, definitions)
                fields.append(T.StructField(name, spark_type, nullable=True))
            return T.StructType(fields)
        for sub in schema_node["allOf"]:
            if isinstance(sub, dict) and ("type" in sub or "properties" in sub):
                return _json_schema_to_spark(sub, definitions)
        return T.StringType()

    for composite_key in ("anyOf", "oneOf"):
        if composite_key in schema_node and not json_type:
            for sub in schema_node[composite_key]:
                if isinstance(sub, dict) and ("type" in sub or "properties" in sub):
                    return _json_schema_to_spark(sub, definitions)
            return T.StringType()

    if json_type == "object":
        props = schema_node.get("properties", {})
        if props:
            fields = []
            for name, prop_schema in props.items():
                spark_type = _json_schema_to_spark(prop_schema, definitions)
                fields.append(T.StructField(name, spark_type, nullable=True))
            return T.StructType(fields)
        return T.MapType(T.StringType(), T.StringType())

    if json_type == "array":
        items = schema_node.get("items", {})
        element_type = _json_schema_to_spark(items, definitions)
        return T.ArrayType(element_type, containsNull=True)

    if json_type in _JSON_TYPE_MAP:
        return _JSON_TYPE_MAP[json_type]

    fmt = schema_node.get("format", "")
    if fmt in ("date-time", "date"):
        return T.StringType()

    return T.StringType()


def _classify_spark_type(dt: T.DataType) -> str:
    if isinstance(dt, T.ArrayType):
        return "json_array"
    if isinstance(dt, (T.StructType, T.MapType)):
        return "json_object"
    return "scalar"


def _parse_osdu_schema(schema_json: str | dict) -> dict[str, Any]:
    raw = json.loads(schema_json) if isinstance(schema_json, str) else schema_json

    definitions = raw.get("definitions", {})
    top_props = dict(raw.get("properties", {}))

    if "allOf" in raw:
        for sub in raw["allOf"]:
            if not isinstance(sub, dict):
                continue
            if "properties" in sub:
                top_props.update(sub["properties"])
            elif "$ref" in sub:
                ref = sub["$ref"]
                if ref.startswith("#/definitions/"):
                    def_key = ref[len("#/definitions/") :]
                    defn = definitions.get(def_key, {})
                    def_props = defn.get("properties", {})
                    top_props.update(def_props)

    envelope_fields = {k: v for k, v in top_props.items() if k != "data"}
    data_node = top_props.get("data", {})
    data_fields = dict(data_node.get("properties", {}))

    if "allOf" in data_node:
        for sub in data_node["allOf"]:
            if not isinstance(sub, dict):
                continue
            if "properties" in sub:
                data_fields.update(sub["properties"])
            elif "$ref" in sub:
                ref = sub["$ref"]
                if ref.startswith("#/definitions/"):
                    def_key = ref[len("#/definitions/") :]
                    defn = definitions.get(def_key, {})
                    def_props = defn.get("properties", {})
                    data_fields.update(def_props)

    return {
        "envelope_fields": envelope_fields,
        "data_fields": data_fields,
        "definitions": definitions,
        "raw": raw,
    }


class SchemaRegistry:
    """In-memory registry of OSDU schemas for schema-driven decomposition."""

    def __init__(self, schemas: dict[str, dict[str, Any]]) -> None:
        self._schemas = schemas
        logger.info("SchemaRegistry loaded: %d kinds", len(schemas))

    @classmethod
    def from_dict(cls, raw_schemas: dict[str, str | dict]) -> "SchemaRegistry":
        schemas = {}
        for kind, schema in raw_schemas.items():
            try:
                parsed = _parse_osdu_schema(schema)
                schemas[kind] = parsed
            except (json.JSONDecodeError, KeyError, TypeError) as e:
                logger.warning("Failed to parse schema for %s: %s", kind, e)
        return cls(schemas)

    @property
    def kinds(self) -> list[str]:
        return list(self._schemas.keys())

    def has_kind(self, kind: str) -> bool:
        return kind in self._schemas

    def has_field(self, kind: str, field_path: str) -> bool:
        info = self._schemas.get(kind)
        if info is None:
            return False
        if field_path in info["envelope_fields"]:
            return True
        clean = field_path[5:] if field_path.startswith("data.") else field_path
        return clean in info["data_fields"]

    def data_fields(self, kind: str) -> dict[str, Any]:
        info = self._schemas.get(kind)
        if info is None:
            raise KeyError(f"Kind {kind!r} not found in registry")
        return info["data_fields"]

    def classify_kind(
        self,
        kind: str,
        actual_columns: list[str] | None = None,
    ) -> dict[str, list[str]]:
        info = self._schemas.get(kind)
        if info is None:
            raise KeyError(f"Kind {kind!r} not found in registry")

        data_fields = info["data_fields"]
        envelope_fields = info["envelope_fields"]

        result: dict[str, list[str]] = {
            "scalar": [],
            "json_array": [],
            "json_object": [],
            "null": [],
        }

        field_lookup: dict[str, dict] = {}

        data_node = info["raw"].get("properties", {}).get("data", {})
        if not data_node and "allOf" in info["raw"]:
            for sub in info["raw"]["allOf"]:
                if isinstance(sub, dict) and "properties" in sub and "data" in sub["properties"]:
                    data_node = sub["properties"]["data"]
                    break
        if data_node:
            field_lookup["data"] = data_node

        field_lookup.update(envelope_fields)

        for name, schema_node in data_fields.items():
            field_lookup[f"data.{name}"] = schema_node
            field_lookup[name] = schema_node

        columns_to_classify = actual_columns if actual_columns is not None else list(field_lookup.keys())

        for col in columns_to_classify:
            schema_node = field_lookup.get(col)

            if schema_node is None and "." in col:
                path = col
                if path.startswith("data."):
                    path = path[5:]
                parts = path.split(".")
                current = data_fields
                resolved = None
                for i, part in enumerate(parts):
                    node = current.get(part)
                    if node is None:
                        break
                    node = _resolve_node(node, info.get("definitions"))
                    if i == len(parts) - 1:
                        resolved = node
                    else:
                        current = node.get("properties", {})
                if resolved is not None:
                    schema_node = resolved

            if schema_node is None:
                result["scalar"].append(col)
                continue

            spark_type = _json_schema_to_spark(schema_node, info.get("definitions"))
            category = _classify_spark_type(spark_type)
            result[category].append(col)

        return result

    def spark_schema_for_field(
        self,
        kind: str,
        field_path: str,
    ) -> T.DataType | None:
        info = self._schemas.get(kind)
        if info is None:
            return None

        if field_path == "data":
            defs = info.get("definitions")
            fields = []
            for name, schema_node in info["data_fields"].items():
                spark_type = _json_schema_to_spark(schema_node, defs)
                fields.append(T.StructField(name, spark_type, nullable=True))
            return T.StructType(fields) if fields else None

        clean_path = field_path
        if clean_path.startswith("data."):
            clean_path = clean_path[5:]

        parts = clean_path.split(".")
        current = info["data_fields"]
        defs = info.get("definitions")
        for i, part in enumerate(parts):
            if part not in current:
                if i == 0 and part in info["envelope_fields"]:
                    current = info["envelope_fields"]
                else:
                    return None

            node = current[part]
            node = _resolve_node(node, defs)
            if i < len(parts) - 1:
                if node.get("type") == "object":
                    current = node.get("properties", {})
                elif node.get("type") == "array":
                    items = node.get("items", {})
                    items = _resolve_node(items, defs)
                    current = items.get("properties", {})
                else:
                    return None
            else:
                return _json_schema_to_spark(node, defs)

        return None

    def spark_struct_for_kind(self, kind: str) -> T.StructType:
        info = self._schemas.get(kind)
        if info is None:
            raise KeyError(f"Kind {kind!r} not found in registry")

        defs = info.get("definitions")
        fields = []
        for name, schema_node in info["data_fields"].items():
            spark_type = _json_schema_to_spark(schema_node, defs)
            fields.append(T.StructField(name, spark_type, nullable=True))

        return T.StructType(fields)


def build_registry_from_web(kinds: list[str]) -> SchemaRegistry:
    """Fetch OSDU schemas via the public web registry and build a SchemaRegistry.

    This is the standalone substitute for SchemaRegistry.from_spark(spark, "osdu_schemas").
    """
    raw: dict[str, dict] = {}
    for kind in kinds:
        try:
            doc, src = load_schema_doc(kind)
            raw[kind] = doc
            logger.info("Loaded schema for %s (%s)", kind, src)
        except Exception as exc:
            logger.warning("Failed to load schema for %s: %s", kind, exc)
    return SchemaRegistry.from_dict(raw)


# ── decompose.py ──────────────────────────────────────────────────────

_FLATTEN_SAMPLE_SIZE = 10
_FLATTEN_PASS_LIMIT = 10
_DATA_PREFIX = "data__"


def _merge_struct_types(a: T.StructType, b: T.StructType) -> T.StructType:
    fields_by_name: dict[str, T.StructField] = {f.name: f for f in a.fields}
    for field in b.fields:
        if field.name not in fields_by_name:
            fields_by_name[field.name] = field
        else:
            existing = fields_by_name[field.name]
            if isinstance(existing.dataType, T.NullType):
                fields_by_name[field.name] = field
            elif isinstance(field.dataType, T.NullType):
                pass
            elif isinstance(existing.dataType, T.StructType) and isinstance(field.dataType, T.StructType):
                merged = _merge_struct_types(existing.dataType, field.dataType)
                fields_by_name[field.name] = T.StructField(field.name, merged, nullable=True)
    return T.StructType(list(fields_by_name.values()))


def _merge_schemas(a: T.DataType, b: T.DataType) -> T.DataType:
    if isinstance(a, T.StructType) and isinstance(b, T.StructType):
        return _merge_struct_types(a, b)
    if isinstance(a, T.ArrayType) and isinstance(b, T.ArrayType):
        merged_elem = _merge_schemas(a.elementType, b.elementType)
        return T.ArrayType(merged_elem, containsNull=True)
    if isinstance(a, T.NullType):
        return b
    return a


def _normalize_json_sample_value(val: str, array_elements: bool = False) -> str | None:
    if not (val and isinstance(val, str) and val.strip() and val.strip()[0] in "{["):
        return None
    try:
        if array_elements:
            parsed = json.loads(val)
            if not (isinstance(parsed, list) and parsed and isinstance(parsed[0], dict)):
                return None
            val = json.dumps(parsed[0])
        json.loads(val)
        return val
    except Exception:
        return None


def _infer_json_schemas_from_values(spark, values: list[str]) -> list[T.DataType | None]:
    if not values:
        return []
    try:
        schema_exprs = [F.schema_of_json(F.lit(v)).alias(f"_schema_{i}") for i, v in enumerate(values)]
        row = spark.range(1).select(*schema_exprs).collect()[0]
        inferred: list[T.DataType | None] = []
        for i in range(len(values)):
            schema_str = row[f"_schema_{i}"]
            inferred.append(T._parse_datatype_string(schema_str) if schema_str else None)
        return inferred
    except Exception:
        return [None] * len(values)


def _infer_json_schema_from_value(spark, val: str, array_elements: bool = False) -> T.DataType | None:
    normalized = _normalize_json_sample_value(val, array_elements=array_elements)
    if normalized is None:
        return None
    return _infer_json_schemas_from_values(spark, [normalized])[0]


def _infer_json_schema(
    df: DataFrame,
    col_name: str,
    sample_size: int = _FLATTEN_SAMPLE_SIZE,
    array_elements: bool = False,
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
    cache_scope: str = "",
) -> T.DataType | None:
    cache_key = (cache_scope, col_name, array_elements)
    if schema_cache is not None and cache_key in schema_cache:
        return schema_cache[cache_key]

    non_null = df.filter(F.col(col_name).isNotNull()).select(col_name)
    samples = non_null.limit(sample_size).collect()
    if not samples:
        if schema_cache is not None:
            schema_cache[cache_key] = None
        return None

    spark = df.sparkSession
    merged: T.DataType | None = None

    normalized_values: list[str] = []
    for row in samples:
        normalized = _normalize_json_sample_value(row[0], array_elements=array_elements)
        if normalized is not None:
            normalized_values.append(normalized)

    if normalized_values:
        inferred_schemas = _infer_json_schemas_from_values(spark, normalized_values)
        for sample_schema in inferred_schemas:
            if sample_schema is None:
                continue
            merged = sample_schema if merged is None else _merge_schemas(merged, sample_schema)

    if schema_cache is not None:
        schema_cache[cache_key] = merged
    return merged


_ENVELOPE_ALIASES: list[tuple[str, str, bool]] = [
    ("createTime", "create_time", True),
    ("modifyTime", "modify_time", True),
    ("createUser", "create_user", False),
    ("modifyUser", "modify_user", False),
]

_ENVELOPE_RENAMES: list[tuple[str, str]] = []  # OSDU-native id/kind/version preserved


def _extract_envelope(df: DataFrame) -> DataFrame:
    cols = set(df.columns)

    for src, tgt in _ENVELOPE_RENAMES:
        if src in cols and tgt not in cols:
            df = df.withColumnRenamed(src, tgt)
            cols.discard(src)
            cols.add(tgt)

    for camel, snake, cast_ts in _ENVELOPE_ALIASES:
        if camel in cols:
            expr = F.col(camel).cast(T.TimestampType()) if cast_ts else F.col(camel)
            df = df.withColumn(snake, expr)
            if camel != snake:
                df = df.drop(camel)
                cols.discard(camel)
                cols.add(snake)
        elif snake in cols and cast_ts:
            df = df.withColumn(snake, F.col(snake).cast(T.TimestampType()))

    df = df.withColumn("ingested_at", F.current_timestamp())
    return df


def classify_columns(
    df: DataFrame,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
) -> dict[str, list[str]]:
    if registry is None or kind is None:
        raise ValueError(
            "SchemaRegistry and kind are required for classify_columns. Sampling-based classification has been removed."
        )

    dot_columns = [c.replace("__", ".") for c in df.columns]
    classified = registry.classify_kind(kind, actual_columns=dot_columns)

    unresolved = [
        col for col in dot_columns if col in classified.get("scalar", []) and not registry.has_field(kind, col)
    ]
    if unresolved:
        logger.warning(
            "Columns not found in registry for kind %r (classified as scalar): %s",
            kind,
            unresolved,
        )

    return {cat: [c.replace(".", "__") for c in cols] for cat, cols in classified.items()}


def flatten_json_object_col(
    df: DataFrame,
    col_name: str,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
) -> DataFrame:
    col_type = df.schema[col_name].dataType
    if not isinstance(col_type, T.StringType):
        logger.warning(
            "Column %r classified as json_object but has type %s — skipping flatten",
            col_name,
            col_type.simpleString(),
        )
        return df

    schema = None
    if registry is not None and kind is not None:
        field_path = col_name.replace("__", ".")
        schema = registry.spark_schema_for_field(kind, field_path)
        if schema is not None and not isinstance(schema, T.StructType):
            logger.debug(
                "Column %r registry-typed %s — will try sampling for struct schema",
                col_name,
                type(schema).__name__,
            )
            schema = None

    if schema is None:
        schema = _infer_json_schema(df, col_name)
        if schema is None or not isinstance(schema, T.StructType):
            logger.debug(
                "Column %r — no struct schema from registry or sampling, deferring to loop",
                col_name,
            )
            return df

    parsed_alias = f"_parsed_{col_name.replace('__', '_')}"
    df = df.withColumn(parsed_alias, F.from_json(F.col(col_name), schema))
    for field in schema.fields:
        new_col = f"{col_name}__{field.name}"
        df = df.withColumn(new_col, F.col(f"{parsed_alias}.{field.name}"))
    df = df.drop(parsed_alias).drop(col_name)
    return df


def build_parent(
    df: DataFrame,
    col_classes: dict[str, list[str]],
    drop_wkt: bool = True,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
) -> DataFrame:
    keep_cols = col_classes["scalar"] + col_classes["null"]
    parent_cols = list(dict.fromkeys(keep_cols + col_classes["json_object"]))
    parent = df.select([F.col(c) for c in parent_cols])

    for obj_col in col_classes["json_object"]:
        if obj_col in parent.columns:
            parent = flatten_json_object_col(parent, obj_col, registry=registry, kind=kind)

    if drop_wkt:
        drop_cols = [c for c in parent.columns if c.endswith("__wkt")]
        if drop_cols:
            parent = parent.drop(*drop_cols)

    parent = _extract_envelope(parent)
    return parent


def _build_child_primitive(df: DataFrame, col_name: str) -> DataFrame:
    schema = T.ArrayType(T.StringType())
    parsed = df.select(
        "id",
        F.from_json(F.col(col_name), schema).alias("_items"),
    ).filter(F.col("_items").isNotNull())
    return parsed.select(
        "id",
        F.posexplode("_items").alias("ordinal", "value"),
    )


def _build_child_struct(df: DataFrame, col_name: str, array_schema: T.ArrayType) -> DataFrame:
    parsed = df.select(
        "id",
        F.from_json(F.col(col_name), array_schema).alias("_items"),
    ).filter(F.col("_items").isNotNull())

    exploded = parsed.select(
        "id",
        F.posexplode("_items").alias("ordinal", "_item"),
    )

    if isinstance(array_schema.elementType, T.StructType):
        select_cols = ["id", "ordinal"]
        select_cols.extend(F.col(f"_item.{fld.name}").alias(fld.name) for fld in array_schema.elementType.fields)
        return _flatten_inferred_json_string_columns(_flatten_all_struct_columns(exploded.select(select_cols)))
    else:
        return exploded.select("id", "ordinal", F.col("_item").alias("value"))


def _is_primitive_array(sample_json: str) -> bool:
    try:
        items = json.loads(sample_json)
        if not isinstance(items, list) or len(items) == 0:
            return False
        return not isinstance(items[0], dict)
    except (json.JSONDecodeError, TypeError):
        return False


def build_child_table(
    df: DataFrame,
    col_name: str,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
    sample_val: str | None = None,
) -> DataFrame:
    spark = df.sparkSession

    if sample_val is None:
        sample_row = df.filter(F.col(col_name).isNotNull()).select(col_name).limit(1).collect()
        if sample_row:
            sample_val = sample_row[0][0]

    if sample_val is None:
        schema = T.StructType(
            [
                T.StructField("id", T.StringType()),
                T.StructField("ordinal", T.IntegerType()),
            ]
        )
        return spark.createDataFrame([], schema)

    if _is_primitive_array(sample_val):
        return _build_child_primitive(df, col_name)

    if registry is None or kind is None:
        logger.info("No registry for %r — inferring schema from sample data", col_name)
        inferred = _infer_json_schema(
            df, col_name, schema_cache=schema_cache, cache_scope=f"child_array:{col_name}",
        )
        if inferred is not None:
            schema = inferred if isinstance(inferred, T.ArrayType) else T.ArrayType(inferred)
        else:
            schema = T.ArrayType(T.StringType())
    else:
        field_path = col_name.replace("__", ".")
        reg_type = registry.spark_schema_for_field(kind, field_path)
        if reg_type is not None and isinstance(reg_type, T.ArrayType):
            schema = reg_type
        elif reg_type is not None:
            schema = T.ArrayType(reg_type)
        else:
            inferred = _infer_json_schema(
                df, col_name, schema_cache=schema_cache, cache_scope=f"child_array:{col_name}",
            )
            if inferred is not None:
                logger.info("Column %r not in registry for %r — inferred schema from data", col_name, kind)
                schema = inferred if isinstance(inferred, T.ArrayType) else T.ArrayType(inferred)
            else:
                logger.warning("Column %r not in registry for %r and inference failed — using StringType", col_name, kind)
                schema = T.ArrayType(T.StringType())

    if not isinstance(schema, T.ArrayType):
        schema = T.ArrayType(schema)

    return _build_child_struct(df, col_name, schema)


def build_all_children(
    df: DataFrame,
    col_classes: dict[str, list[str]],
    kind_prefix: str,
    registry: SchemaRegistry | None = None,
    osdu_kind: str | None = None,
) -> dict[str, tuple[str, DataFrame]]:
    children: dict[str, tuple[str, DataFrame]] = {}
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] = {}

    json_array_cols_present = [c for c in col_classes["json_array"] if c in df.columns]
    sampled_values: dict[str, str | None] = {}
    has_items_by_col: dict[str, bool] = {}
    if json_array_cols_present:
        sample_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in json_array_cols_present]
        sample_row = df.select(sample_exprs).collect()
        if sample_row:
            row = sample_row[0]
            sampled_values = {c: row[c] for c in json_array_cols_present}
        else:
            sampled_values = {c: None for c in json_array_cols_present}

        has_items_by_col = _batch_sample_json_array_presence(df, json_array_cols_present)

    for col_name in col_classes["json_array"]:
        if col_name in has_items_by_col and not has_items_by_col[col_name]:
            logger.info("Skipping empty json_array child %r (no non-empty arrays in batch)", col_name)
            continue

        normalized = col_name.replace(".", "__")
        if normalized.startswith(_DATA_PREFIX):
            normalized = normalized[len(_DATA_PREFIX) :]
        parts = [p for p in normalized.split("__") if p]
        table_suffix = "__".join(parts[-2:]) if len(parts) > 2 else "__".join(parts)

        table_name = child_table_name(kind_prefix, table_suffix)
        child_df = build_child_table(
            df, col_name, registry=registry, kind=osdu_kind,
            schema_cache=schema_cache, sample_val=sampled_values.get(col_name),
        )
        children[table_name] = (col_name, child_df)

    return children


_TAGS_COLUMN = "tags"


def extract_tags(df: DataFrame, kind_prefix: str) -> tuple[str, DataFrame] | None:
    if _TAGS_COLUMN not in df.columns:
        return None

    schema = T.MapType(T.StringType(), T.StringType())
    parsed = df.select(
        "id",
        F.from_json(F.col(_TAGS_COLUMN), schema).alias("_tags_map"),
    ).filter(F.col("_tags_map").isNotNull())

    if not parsed.take(1):
        return None

    child = parsed.select(
        "id",
        F.explode("_tags_map").alias("tag_key", "tag_value"),
    )

    table_name = child_table_name(kind_prefix, "tags")
    return table_name, child


_NUMERIC_TYPES = (T.DoubleType, T.FloatType, T.LongType, T.IntegerType, T.ShortType)


def _coerce_id_columns(df: DataFrame) -> DataFrame:
    for field in df.schema.fields:
        if field.name.endswith("_id") and isinstance(field.dataType, _NUMERIC_TYPES):
            df = df.withColumn(field.name, F.col(field.name).cast(T.StringType()))
    return df


def _serialize_complex_columns(df: DataFrame) -> DataFrame:
    for field in df.schema.fields:
        if isinstance(field.dataType, T.MapType) or (
            isinstance(field.dataType, T.ArrayType) and not isinstance(field.dataType.elementType, T.StructType)
        ):
            df = df.withColumn(field.name, F.to_json(F.col(field.name)))
    return df


def _explode_typed_array(
    df: DataFrame,
    col_name: str,
    elem_type: T.DataType,
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
) -> DataFrame:
    subset = df.select("id", F.col(col_name)).filter(F.col(col_name).isNotNull())
    exploded = subset.select(
        "id",
        F.posexplode(F.col(col_name)).alias("ordinal", "_item"),
    )

    if isinstance(elem_type, T.StructType):
        select_cols = ["id", "ordinal"]
        select_cols.extend(F.col(f"_item.{fld.name}").alias(fld.name) for fld in elem_type.fields)
        return _flatten_inferred_json_string_columns(_flatten_all_struct_columns(exploded.select(select_cols)))

    if isinstance(elem_type, T.StringType):
        child = exploded.select("id", "ordinal", F.col("_item").alias("value"))
        inferred = _infer_json_schema(
            child, "value", schema_cache=schema_cache, cache_scope=f"typed_array:{col_name}",
        )
        if inferred is not None and isinstance(inferred, T.StructType):
            logger.info(
                "Column %r has StringType elements containing JSON objects — re-parsing with inferred schema (%d fields: %s)",
                col_name, len(inferred.fields), ", ".join(f.name for f in inferred.fields),
            )
            child = child.withColumn("_parsed", F.from_json(F.col("value"), inferred))
            select_cols: list = ["id", "ordinal"]
            for fld in inferred.fields:
                select_cols.append(F.col(f"_parsed.{fld.name}").alias(fld.name))
            return _flatten_inferred_json_string_columns(_flatten_all_struct_columns(child.select(select_cols)))
        return child

    return exploded.select("id", "ordinal", F.col("_item").alias("value"))


def _batch_sample_string_arrays(
    parent: DataFrame,
    fields: list[T.StructField],
) -> dict[str, bool]:
    if not fields:
        return {}

    sample_exprs = [
        F.first(
            F.when(F.size(F.col(f.name)) > 0, F.col(f.name).getItem(0)),
            ignorenulls=True,
        ).alias(f.name)
        for f in fields
    ]

    samples = parent.select(sample_exprs).collect()
    if not samples:
        return {f.name: False for f in fields}

    row = samples[0]
    results: dict[str, bool] = {}
    for field in fields:
        val = row[field.name]
        if not val or not isinstance(val, str):
            results[field.name] = False
            continue
        try:
            parsed = json.loads(val)
            results[field.name] = isinstance(parsed, dict)
        except (json.JSONDecodeError, ValueError):
            results[field.name] = False

    return results


def _batch_sample_array_presence(
    parent: DataFrame,
    fields: list[T.StructField],
) -> dict[str, bool]:
    if not fields:
        return {}

    sample_exprs = [
        F.first(
            F.when(F.size(F.col(f.name)) > 0, F.lit(1)),
            ignorenulls=True,
        ).alias(f.name)
        for f in fields
    ]

    samples = parent.select(sample_exprs).collect()
    if not samples:
        return {f.name: False for f in fields}

    row = samples[0]
    return {field.name: row[field.name] is not None for field in fields}


def _batch_sample_json_array_presence(
    df: DataFrame,
    columns: list[str],
) -> dict[str, bool]:
    if not columns:
        return {}

    sample_exprs = [
        F.first(
            F.when(
                F.col(c).isNotNull() & (F.length(F.regexp_replace(F.col(c), r"\s+", "")) > 2),
                F.lit(1),
            ),
            ignorenulls=True,
        ).alias(c)
        for c in columns
    ]

    samples = df.select(sample_exprs).collect()
    if not samples:
        return {c: False for c in columns}

    row = samples[0]
    return {c: row[c] is not None for c in columns}


def _extract_typed_arrays(
    parent: DataFrame,
    kind_prefix: str,
    retained_parent_arrays: set[str] | None = None,
    schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
) -> tuple[dict[str, tuple[str, DataFrame]], DataFrame, set[str]]:
    children: dict[str, tuple[str, DataFrame]] = {}
    retained_parent_arrays = set() if retained_parent_arrays is None else retained_parent_arrays

    array_cols = [f for f in parent.schema.fields if isinstance(f.dataType, T.ArrayType)]

    string_array_fields = [
        f
        for f in array_cols
        if f.name not in retained_parent_arrays and isinstance(f.dataType.elementType, T.StringType)
    ]
    string_array_results = _batch_sample_string_arrays(parent, string_array_fields)

    extract_fields: list[T.StructField] = []
    candidate_extract_fields: list[T.StructField] = []
    for field in array_cols:
        if field.name in retained_parent_arrays:
            continue

        elem_type = field.dataType.elementType
        if isinstance(elem_type, T.StructType):
            is_child = True
        elif isinstance(elem_type, T.StringType):
            is_child = string_array_results.get(field.name, False)
        else:
            is_child = False

        if not is_child:
            retained_parent_arrays.add(field.name)
            logger.info(
                "Keeping primitive array %r on parent (element type: %s)",
                field.name, field.dataType.elementType.simpleString(),
            )
            continue

        candidate_extract_fields.append(field)

    has_items_by_col = _batch_sample_array_presence(parent, candidate_extract_fields)

    for field in candidate_extract_fields:
        if not has_items_by_col.get(field.name, False):
            logger.info("Skipping empty typed array %r (no non-empty values in batch)", field.name)
            continue

        extract_fields.append(field)
        col_name = field.name
        normalized = col_name
        if normalized.startswith(_DATA_PREFIX):
            normalized = normalized[len(_DATA_PREFIX) :]
        parts = [p for p in normalized.split("__") if p]
        table_suffix = "__".join(parts[-2:]) if len(parts) > 2 else "__".join(parts)

        table_name = child_table_name(kind_prefix, table_suffix)
        child_df = _explode_typed_array(parent, col_name, field.dataType.elementType, schema_cache=schema_cache)
        children[table_name] = (col_name, child_df)

    if extract_fields:
        parent = parent.drop(*[f.name for f in extract_fields])

    return children, parent, retained_parent_arrays


def _flatten_typed_structs(parent: DataFrame) -> DataFrame:
    struct_fields = [f for f in parent.schema.fields if isinstance(f.dataType, T.StructType)]
    if not struct_fields:
        return parent

    select_exprs = []
    struct_names = {f.name for f in struct_fields}
    for field in parent.schema.fields:
        if field.name in struct_names:
            prefix = field.name
            select_exprs.extend(
                F.col(f"{prefix}.{sub.name}").alias(f"{prefix}__{sub.name}") for sub in field.dataType.fields
            )
        else:
            select_exprs.append(F.col(field.name))

    return parent.select(select_exprs)


def _flatten_all_struct_columns(df: DataFrame) -> DataFrame:
    while any(isinstance(field.dataType, T.StructType) for field in df.schema.fields):
        df = _flatten_typed_structs(df)
    return df


def _flatten_inferred_json_string_columns(df: DataFrame) -> DataFrame:
    while True:
        string_cols = [field.name for field in df.schema.fields if isinstance(field.dataType, T.StringType)]
        if not string_cols:
            return df

        sample_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in string_cols]
        samples = df.select(sample_exprs).collect()
        row = samples[0] if samples else None

        flattened_any = False

        normalized_by_col: dict[str, str] = {}
        for col_name in string_cols:
            val = row[col_name] if row is not None else None
            normalized = _normalize_json_sample_value(val)
            if normalized is not None:
                normalized_by_col[col_name] = normalized

        inferred_by_col: dict[str, T.DataType | None] = {}
        if normalized_by_col:
            cols = list(normalized_by_col.keys())
            vals = [normalized_by_col[c] for c in cols]
            inferred = _infer_json_schemas_from_values(df.sparkSession, vals)
            inferred_by_col = {c: s for c, s in zip(cols, inferred, strict=False)}

        for col_name in string_cols:
            inferred = inferred_by_col.get(col_name)
            if inferred is None or not isinstance(inferred, T.StructType):
                continue

            logger.info(
                "Column %r contains embedded JSON objects — flattening inferred schema (%d fields: %s)",
                col_name, len(inferred.fields), ", ".join(field.name for field in inferred.fields),
            )
            df = df.withColumn("_parsed", F.from_json(F.col(col_name), inferred))
            for field in inferred.fields:
                df = df.withColumn(f"{col_name}__{field.name}", F.col(f"_parsed.{field.name}"))
            df = df.drop(col_name, "_parsed")
            flattened_any = True
            break

        if not flattened_any:
            return df


def _flatten_remaining_json_objects(
    parent: DataFrame,
    registry: SchemaRegistry | None = None,
    kind: str | None = None,
    _sampled_cache: dict[str, str | None] | None = None,
    _schema_cache: dict[tuple[str, str, bool], T.DataType | None] | None = None,
) -> tuple[DataFrame, bool]:
    string_cols = [f.name for f in parent.schema.fields if isinstance(f.dataType, T.StringType)]
    if not string_cols:
        return parent, False

    if _sampled_cache is None:
        _sampled_cache = {}

    new_cols = [c for c in string_cols if c not in _sampled_cache]
    if new_cols:
        sample_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in new_cols]
        samples = parent.select(sample_exprs).collect()
        if samples:
            row = samples[0]
            for col_name in new_cols:
                _sampled_cache[col_name] = row[col_name]
        else:
            for col_name in new_cols:
                _sampled_cache[col_name] = None

    json_obj_cols = []
    json_array_cols = []
    for col_name in string_cols:
        val = _sampled_cache.get(col_name)
        if not val or not isinstance(val, str):
            continue
        stripped = val.strip()
        if stripped.startswith("{"):
            try:
                json.loads(val)
                json_obj_cols.append(col_name)
            except (json.JSONDecodeError, ValueError):
                pass
        elif stripped.startswith("[{"):
            try:
                parsed = json.loads(val)
                if parsed and isinstance(parsed[0], dict):
                    json_array_cols.append(col_name)
            except (json.JSONDecodeError, ValueError):
                pass

    if not json_obj_cols and not json_array_cols:
        return parent, False

    flattened_json_objects = False

    cols_to_drop = []
    parsed_col_defs: list[tuple[str, str, T.StructType]] = []

    for col_name in json_obj_cols:
        schema = None
        sampled_val = _sampled_cache.get(col_name)
        if registry is not None and kind is not None:
            field_path = col_name.replace("__", ".")
            schema = registry.spark_schema_for_field(kind, field_path)

        if schema is not None and not isinstance(schema, T.StructType):
            logger.debug(
                "Column %r is registry-typed %s but contains JSON objects — inferring schema from data",
                col_name, type(schema).__name__,
            )
            schema = _infer_json_schema_from_value(parent.sparkSession, sampled_val)

        if schema is None:
            schema = _infer_json_schema_from_value(parent.sparkSession, sampled_val)

        if schema is None or not isinstance(schema, T.StructType):
            logger.warning(
                "Nested JSON in %r not resolvable from registry or sampling for kind %r — skipping",
                col_name, kind,
            )
            continue

        safe_suffix = col_name.split("__")[-1]
        parsed_alias = f"_parsed_{safe_suffix}_{len(parsed_col_defs)}"
        parsed_col_defs.append((col_name, parsed_alias, schema))
        cols_to_drop.append(col_name)

    if parsed_col_defs:
        for col_name, parsed_alias, schema in parsed_col_defs:
            parent = parent.withColumn(parsed_alias, F.from_json(F.col(col_name), schema))

        select_exprs = []
        drop_set = set(cols_to_drop) | {alias for _, alias, _ in parsed_col_defs}
        select_exprs.extend(F.col(field.name) for field in parent.schema.fields if field.name not in drop_set)

        for col_name, parsed_alias, schema in parsed_col_defs:
            for field in schema.fields:
                new_col = f"{col_name}__{field.name}"
                select_exprs.append(F.col(f"{parsed_alias}.{field.name}").alias(new_col))

        parent = parent.select(select_exprs)
        flattened_json_objects = True

    for col_name in json_array_cols:
        schema = None
        sampled_val = _sampled_cache.get(col_name)
        if registry is not None and kind is not None:
            field_path = col_name.replace("__", ".")
            schema = registry.spark_schema_for_field(kind, field_path)

        elem_schema = None
        if schema is not None and isinstance(schema, T.ArrayType):
            if isinstance(schema.elementType, T.StructType):
                elem_schema = schema.elementType
        elif schema is not None and isinstance(schema, T.StructType):
            elem_schema = schema

        if elem_schema is None:
            inferred = _infer_json_schema_from_value(parent.sparkSession, sampled_val, array_elements=True)
            if inferred is not None and isinstance(inferred, T.StructType):
                elem_schema = inferred

        if elem_schema is None:
            logger.warning(
                "JSON array in %r not resolvable from registry or sampling for kind %r — skipping",
                col_name, kind,
            )
            continue

        array_schema = T.ArrayType(elem_schema)
        parent = parent.withColumn(col_name, F.from_json(F.col(col_name), array_schema))
        logger.info(
            "Parsed JSON array string %r → ArrayType(StructType(%d fields))",
            col_name, len(elem_schema.fields),
        )
        flattened_json_objects = True

    return parent, flattened_json_objects


def decompose_kind(
    df: DataFrame,
    kind_name: str,
    drop_wkt: bool = True,
    registry: SchemaRegistry | None = None,
    osdu_kind: str | None = None,
    input_rows: int | None = None,
    infer_nested_json: bool = True,
) -> DecomposedKind:
    t_start = perf_counter()
    stage_times: dict[str, float] = {}

    def _record_stage(name: str, started_at: float) -> None:
        stage_times[name] = stage_times.get(name, 0.0) + (perf_counter() - started_at)

    t_stage = perf_counter()
    col_classes = classify_columns(df, registry=registry, kind=osdu_kind)
    _record_stage("classify_columns", t_stage)
    row_info = str(input_rows) if input_rows is not None else "unknown"
    logger.info(
        "Decomposing kind=%s  rows=%s  cols=%d  (scalar=%d, json_array=%d, json_object=%d, null=%d)",
        kind_name, row_info, len(df.columns),
        len(col_classes["scalar"]), len(col_classes["json_array"]),
        len(col_classes["json_object"]), len(col_classes["null"]),
    )

    t_stage = perf_counter()
    if registry is not None:
        json_cols = col_classes["json_object"] + col_classes["json_array"]
        for col_name in json_cols:
            if col_name in df.columns:
                col_type = df.schema[col_name].dataType
                if not isinstance(col_type, T.StringType):
                    logger.info(
                        "Casting %r from %s → StringType (registry says json)",
                        col_name, col_type.simpleString(),
                    )
                    df = df.withColumn(col_name, F.col(col_name).cast(T.StringType()))
    _record_stage("coerce_json_columns", t_stage)

    if _TAGS_COLUMN in col_classes["json_object"]:
        col_classes["json_object"].remove(_TAGS_COLUMN)
    if _TAGS_COLUMN in col_classes["scalar"]:
        col_classes["scalar"].remove(_TAGS_COLUMN)

    t_stage = perf_counter()
    json_array_cols_present = [c for c in col_classes["json_array"] if c in df.columns]
    if json_array_cols_present:
        sample_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in json_array_cols_present]
        sample_row = df.select(sample_exprs).collect()
        primitive_arrays: list[str] = []
        if sample_row:
            row = sample_row[0]
            for col_name in json_array_cols_present:
                val = row[col_name]
                if val and _is_primitive_array(val):
                    primitive_arrays.append(col_name)
        if primitive_arrays:
            col_classes["json_array"] = [c for c in col_classes["json_array"] if c not in primitive_arrays]
            col_classes["scalar"].extend(primitive_arrays)
            logger.info(
                "Reclassified %d primitive array column(s) as scalar: %s",
                len(primitive_arrays), primitive_arrays,
            )
    _record_stage("reclassify_primitive_arrays", t_stage)

    t_stage = perf_counter()
    parent = build_parent(df, col_classes, drop_wkt=drop_wkt, registry=registry, kind=osdu_kind)
    _record_stage("build_parent", t_stage)

    t_stage = perf_counter()
    children = build_all_children(df, col_classes, kind_prefix=kind_name, registry=registry, osdu_kind=osdu_kind)
    _record_stage("build_all_children", t_stage)
    retained_parent_arrays: set[str] = set()

    _sampled_cache: dict[str, str | None] = {}
    _schema_cache: dict[tuple[str, str, bool], T.DataType | None] = {}

    t_stage = perf_counter()
    typed_children, parent, retained_parent_arrays = _extract_typed_arrays(
        parent, kind_name,
        retained_parent_arrays=retained_parent_arrays,
        schema_cache=_schema_cache,
    )
    _record_stage("extract_typed_arrays_initial", t_stage)
    children.update(typed_children)

    _seen_array_cols: set[str] = set(f.name for f in parent.schema.fields if isinstance(f.dataType, T.ArrayType))

    reached_flatten_pass_limit = False
    pass_index = 0
    while True:
        t_iter = perf_counter()
        made_progress = False

        t_struct = perf_counter()
        had_structs = any(isinstance(f.dataType, T.StructType) for f in parent.schema.fields)
        parent = _flatten_typed_structs(parent)
        _record_stage("flatten_typed_structs", t_struct)
        if had_structs:
            made_progress = True

        if infer_nested_json:
            t_json = perf_counter()
            parent, flattened_json = _flatten_remaining_json_objects(
                parent, registry=registry, kind=osdu_kind,
                _sampled_cache=_sampled_cache, _schema_cache=_schema_cache,
            )
            _record_stage("flatten_remaining_json_objects", t_json)
            if flattened_json:
                made_progress = True

        current_array_cols = set(f.name for f in parent.schema.fields if isinstance(f.dataType, T.ArrayType))
        new_array_cols = current_array_cols - _seen_array_cols
        _seen_array_cols = current_array_cols

        if new_array_cols:
            t_new_arrays = perf_counter()
            new_children, parent, retained_parent_arrays = _extract_typed_arrays(
                parent, kind_name,
                retained_parent_arrays=retained_parent_arrays,
                schema_cache=_schema_cache,
            )
            _record_stage("extract_typed_arrays_loop", t_new_arrays)
            if new_children:
                children.update(new_children)
                made_progress = True

        iter_elapsed = perf_counter() - t_iter
        pass_index += 1
        logger.info(
            "Flatten iteration %d/%d for %s took %.3fs (progress=%s, new_arrays=%d)",
            pass_index, _FLATTEN_PASS_LIMIT, kind_name, iter_elapsed, made_progress, len(new_array_cols),
        )
        stage_times[f"flatten_loop_iter_{pass_index}"] = iter_elapsed

        if not made_progress:
            break

        if pass_index >= _FLATTEN_PASS_LIMIT:
            reached_flatten_pass_limit = True
            break

    if reached_flatten_pass_limit:
        logger.warning(
            "Reached flatten_pass_limit=%d for %s while changes were still being made.",
            _FLATTEN_PASS_LIMIT, kind_name,
        )

    t_stage = perf_counter()
    tags_result = extract_tags(df, kind_name)
    _record_stage("extract_tags", t_stage)
    if tags_result is not None:
        table_name, tags_df = tags_result
        children[table_name] = (_TAGS_COLUMN, tags_df)

    t_stage = perf_counter()
    if drop_wkt:
        drop_cols = [c for c in parent.columns if c.endswith("__wkt")]
        if drop_cols:
            parent = parent.drop(*drop_cols)
    _record_stage("drop_wkt", t_stage)

    t_stage = perf_counter()
    parent = _coerce_id_columns(parent)
    children = {name: (src_col, _coerce_id_columns(child_df)) for name, (src_col, child_df) in children.items()}
    _record_stage("coerce_id_columns", t_stage)

    t_stage = perf_counter()
    parent = _serialize_complex_columns(parent)
    _record_stage("serialize_complex_columns", t_stage)

    t_stage = perf_counter()
    if children:
        probe_frames: list[DataFrame] = []
        for name, (_, child_df) in children.items():
            probe_frames.append(child_df.select(F.lit(name).alias("__child_name")).limit(1))

        non_empty_names: set[str] = set()
        if probe_frames:
            probe_union = reduce(lambda left, right: left.unionByName(right), probe_frames)
            non_empty_names = {row["__child_name"] for row in probe_union.collect()}

        empty_children = [name for name in children if name not in non_empty_names]
        for name in empty_children:
            logger.info("Dropping empty child table %r (0 rows after explode)", name)
        children = {name: v for name, v in children.items() if name in non_empty_names}
    _record_stage("prune_empty_children", t_stage)

    logger.info(
        "Decomposed %s → parent (%d cols) + %d child tables",
        kind_name, len(parent.columns), len(children),
    )
    total_elapsed = perf_counter() - t_start
    logger.info("Decompose timing total for %s: %.3fs", kind_name, total_elapsed)
    for name, elapsed in sorted(stage_times.items(), key=lambda kv: kv[1], reverse=True):
        logger.info("  decompose stage %-32s %.3fs", name, elapsed)

    return DecomposedKind(
        kind_name=kind_name,
        col_classes=col_classes,
        parent=parent,
        children=children,
    )


# ── reassemble.py ─────────────────────────────────────────────────────


def _child_type(child_df: DataFrame) -> str:
    cols = set(child_df.columns)
    if cols == {"id", "tag_key", "tag_value"}:
        return "tags"
    if cols == {"id", "ordinal", "value"}:
        return "primitive"
    return "struct"


def _child_suffix(table_name: str) -> str:
    if "___" in table_name:
        return table_name.split("___", 1)[1]
    return table_name


def _reassemble_tags(parent: DataFrame, child_df: DataFrame) -> DataFrame:
    pivoted = child_df.groupBy("id").pivot("tag_key").agg(F.first("tag_value"))
    for col_name in pivoted.columns:
        if col_name != "id":
            pivoted = pivoted.withColumnRenamed(col_name, f"tag_{col_name}")
    return parent.join(pivoted, on="id", how="left")


def _reassemble_primitive(parent: DataFrame, child_df: DataFrame, suffix: str) -> DataFrame:
    col_alias = suffix.replace("__", "_")
    agg_df = child_df.groupBy("id").agg(F.concat_ws(";", F.collect_list("value")).alias(col_alias))
    return parent.join(agg_df, on="id", how="left")


def _reassemble_struct(
    parent: DataFrame,
    child_df: DataFrame,
    suffix: str,
    max_cardinality_cap: int,
) -> DataFrame:
    max_card_row = child_df.groupBy("id").count().agg(F.max("count").alias("max_count")).collect()
    max_card = max_card_row[0]["max_count"] if max_card_row else 0
    if max_card is None:
        max_card = 0

    if max_card > max_cardinality_cap:
        logger.warning("Child '%s' has max cardinality %d, capping at %d", suffix, max_card, max_cardinality_cap)
        max_card = max_cardinality_cap

    value_fields = [c for c in child_df.columns if c not in ("id", "ordinal")]

    for i in range(max_card):
        ordinal_slice = child_df.filter(F.col("ordinal") == i)
        select_exprs = [F.col("id")]
        for fld in value_fields:
            alias = f"{suffix}_{i}_{fld}"
            select_exprs.append(F.col(fld).alias(alias))
        ordinal_df = ordinal_slice.select(select_exprs)
        parent = parent.join(ordinal_df, on="id", how="left")

    return parent


def reassemble_kind(
    dk: DecomposedKind,
    max_cardinality_cap: int = 20,
) -> DataFrame:
    """Reassemble a DecomposedKind into a single flat DataFrame."""
    result = dk.parent

    for table_name, (source_col, child_df) in dk.children.items():
        suffix = _child_suffix(table_name)
        ctype = _child_type(child_df)

        if ctype == "tags":
            result = _reassemble_tags(result, child_df)
        elif ctype == "primitive":
            result = _reassemble_primitive(result, child_df, suffix)
        elif ctype == "struct":
            result = _reassemble_struct(result, child_df, suffix, max_cardinality_cap)

    return result


# ── Diagnostic schema-walk helpers ────────────────────────────────
# These helpers are not used by decompose_kind; keep them for optional schema inspection.


def collect_schema_fields(kind: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Walk the schema for ``kind``, returning (field_rows, ref_rows).

    Kept for diagnostic compatibility. The runtime decomposition path uses
    SchemaRegistry exclusively.
    """
    field_rows: list[dict[str, Any]] = []
    ref_rows: list[dict[str, Any]] = []
    visited: set[tuple[str, str]] = set()
    queue: deque[tuple[str, dict[str, Any], str]] = deque()

    root_doc, root_src = load_schema_doc(kind)
    root_name = root_doc.get("title") or parse_kind(kind)["entity_base"]
    queue.append((root_name, root_doc, root_src))

    def walk_properties(schema_name: str, schema_doc: dict[str, Any], path_prefix: str) -> None:
        properties = schema_doc.get("properties", {}) or {}
        for prop_name, spec in properties.items():
            if not isinstance(spec, dict):
                continue
            path = f"{path_prefix}.{prop_name}" if path_prefix else prop_name
            dtype = spec.get("type") or ("object" if "properties" in spec else "")
            field_rows.append(
                {
                    "schema": schema_name,
                    "field_path": path,
                    "type": dtype,
                    "resolved_type": dtype,
                }
            )
            if isinstance(spec.get("$ref"), str):
                nref = normalize_kind_ref(spec["$ref"])
                if nref:
                    ref_rows.append(
                        {
                            "from_schema": schema_name,
                            "field_path": path,
                            "normalized_ref": nref,
                            "resolved": False,
                            "source": "",
                        }
                    )
            if "allOf" in spec:
                for i, part in enumerate(spec["allOf"]):
                    if isinstance(part, dict) and part.get("$ref"):
                        nref = normalize_kind_ref(part["$ref"])
                        if nref:
                            ref_rows.append(
                                {
                                    "from_schema": schema_name,
                                    "field_path": f"{path}.allOf[{i}]",
                                    "normalized_ref": nref,
                                    "resolved": False,
                                    "source": "",
                                }
                            )
            if dtype == "object" and "properties" in spec:
                walk_properties(schema_name, spec, path)

    while queue:
        schema_name, schema_doc, schema_source = queue.popleft()
        schema_key = (schema_name, schema_source)
        if schema_key in visited:
            continue
        visited.add(schema_key)

        walk_properties(schema_name, schema_doc, "")

        for r in [x for x in ref_rows if x["from_schema"] == schema_name and not x["resolved"]]:
            try:
                child_doc, child_src = load_schema_doc(r["normalized_ref"])
                r["resolved"] = True
                r["source"] = child_src
                child_name = child_doc.get("title") or parse_kind(r["normalized_ref"])["entity_base"]
                queue.append((child_name, child_doc, child_src))
            except Exception:
                r["resolved"] = False

    return field_rows, ref_rows


print("Standalone Silver Layer decompose/reassemble loaded.")


## Pipeline functions

Load orchestration functions for processing one or more OSDU kinds.

- `process_kind()` reads bronze records, resolves schemas, transforms the data, and writes Silver Layer output for one kind.
- `run_silver_build()` runs `process_kind()` for all configured kinds and records status in `silver_run_info`.


In [ ]:
import re

def _table_exists(spark: SparkSession, table_name: str) -> bool:
    try:
        if spark.catalog.tableExists(table_name):
            return True
    except Exception as exc:
        if not _requires_path_fallback(exc):
            raise

    try:
        from delta.tables import DeltaTable

        return DeltaTable.isDeltaTable(spark, _table_path_uri(table_name))
    except Exception as exc:
        logger.info("Delta table %r was not found through catalog or OneLake path: %s", table_name, exc)
        return False


def _cast_null_columns(df: DataFrame) -> DataFrame:
    # Keep no-op for standalone compatibility
    return df


_TABLE_NAME_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def validate_table_names(table_names: list[str]) -> None:
    invalid = [name for name in table_names if not _TABLE_NAME_PATTERN.match(name or "")]
    if invalid:
        raise ValueError(
            "Invalid output table name(s): "
            + ", ".join(invalid)
            + ". Use letters, numbers, and underscores, and start with a letter or underscore."
        )


def _existing_tables(spark: SparkSession, table_names: list[str]) -> list[str]:
    return [name for name in table_names if _table_exists(spark, name)]


def _assert_overwrite_allowed(spark: SparkSession, table_names: list[str], allow_overwrite: bool) -> None:
    existing = _existing_tables(spark, table_names)
    if existing and not allow_overwrite:
        raise RuntimeError(
            "Full refresh would overwrite existing table(s): "
            + ", ".join(existing)
            + ". Set ALLOW_OVERWRITE = True or ADME_ALLOW_OVERWRITE=true to proceed."
        )


def _delta_table_for_target(spark: SparkSession, target: str):
    from delta.tables import DeltaTable

    try:
        return DeltaTable.forName(spark, target)
    except Exception as exc:
        if not _requires_path_fallback(exc):
            raise

    return DeltaTable.forPath(spark, _table_path_uri(target))


def _incremental_child_write(
    spark: SparkSession,
    new_child_df: DataFrame,
    child_uri: str,
    changed_ids_df: DataFrame,
) -> None:
    """Replace child rows for changed parent ids, then append the new child rows."""
    if not _table_exists(spark, child_uri):
        write_silver_table(new_child_df, child_uri, mode="overwrite")
        return

    changed_ids_df = changed_ids_df.select(F.col("id").cast("string").alias("id")).distinct()
    target = _delta_table_for_target(spark, child_uri)
    (
        target.alias("target")
        .merge(changed_ids_df.alias("changed"), "target.id = changed.id")
        .whenMatchedDelete()
        .execute()
    )

    new_child_df = _cast_null_columns(new_child_df)
    _write_table(new_child_df, child_uri, mode="append")


# ── Run-info metadata ───────────────────────────────────────────────


def write_run_info(
    spark: SparkSession,
    run_id: str,
    kind: str,
    start_time: datetime,
    end_time: datetime,
    records_processed: int,
    records_failed: int,
    status: str,
    error_message: str | None,
    workspace_id: str,
    lakehouse_id: str,
) -> None:
    """Append a run_info record to the runtime-appropriate destination."""
    row = [
        (
            run_id,
            kind,
            start_time,
            end_time,
            records_processed,
            records_failed,
            status,
            error_message,
        )
    ]
    df = spark.createDataFrame(row, schema=RUN_INFO_SCHEMA)
    run_info_uri = table_uri(workspace_id, lakehouse_id, "silver_run_info")

    if _table_exists(spark, run_info_uri):
        _write_table(df, run_info_uri, mode="append")
    else:
        _write_table(df, run_info_uri, mode="overwrite")

    logger.info("Recorded run_info: %s %s → %s", run_id, kind, status)


def write_run_manifest(
    spark: SparkSession,
    run_id: str,
    result: KindResult,
    output_mode: str,
    table_prefix: str,
    bronze_table: str,
    run_profile: str,
    notebook_version: str,
    config_hash: str,
    allow_overwrite: bool,
    workspace_id: str,
    lakehouse_id: str,
) -> None:
    """Append a run manifest row that describes output tables produced for one kind."""
    child_tables = result.child_tables or []
    row = [
        (
            run_id,
            result.kind,
            output_mode,
            notebook_version,
            run_profile,
            config_hash,
            allow_overwrite,
            table_prefix,
            bronze_table,
            result.parent_table,
            child_tables,
            result.records_processed,
            result.status,
            result.error,
            datetime.now(UTC),
        )
    ]
    df = spark.createDataFrame(row, schema=RUN_MANIFEST_SCHEMA)
    manifest_uri = table_uri(workspace_id, lakehouse_id, globals().get("run_manifest_table", "silver_run_manifest"))

    if _table_exists(spark, manifest_uri):
        _write_table(df, manifest_uri, mode="append")
    else:
        _write_table(df, manifest_uri, mode="overwrite")

    logger.info("Recorded run manifest: %s %s", run_id, result.kind)


def process_kind(
    spark: SparkSession,
    kind: str,
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    limit: int | None = None,
    incremental: bool = False,
    drop_wkt: bool = True,
    reassemble: bool = False,
    table_prefix: str = "",
    allow_overwrite: bool = False,
) -> KindResult:
    """Process one OSDU kind from bronze to silver tables.

    Uses the standalone verbatim port of decompose/reassemble with a SchemaRegistry
    built from the OSDU web schema repo (substitute for SchemaRegistry.from_spark).
    """
    kind_name = kind_to_table_name(kind)
    parent_table = f"{table_prefix}{kind_name}"
    validate_table_names([parent_table])

    bronze_df = read_bronze_kind_spark(
        kind,
        spark,
        workspace_id,
        lakehouse_id,
        bronze_table=bronze_table,
    )
    if limit:
        bronze_df = bronze_df.limit(limit)

    records_processed = bronze_df.count()
    if records_processed == 0:
        return KindResult(
            kind=kind,
            status="skipped",
            records_processed=0,
            parent_table=parent_table,
            child_tables=[],
            reassembled=reassemble,
            validation_passed=True,
        )

    # Build SchemaRegistry from the OSDU web schema repo (standalone substitute
    # for SchemaRegistry.from_spark used in the packaged build).
    registry = build_registry_from_web([kind])
    if not registry.has_kind(kind):
        raise RuntimeError(
            f"Schema for {kind!r} could not be loaded from OSDU web; aborting."
        )

    decomposed = decompose_kind(
        bronze_df,
        kind_name=parent_table,
        drop_wkt=drop_wkt,
        registry=registry,
        osdu_kind=kind,
        input_rows=records_processed,
        infer_nested_json=True,
    )

    parent_df = decomposed.parent
    children = decomposed.children

    # Fail fast if JSON-heavy payload was not flattened.
    if "data" in bronze_df.columns:
        has_data_flattened = any(c.startswith("data__") for c in parent_df.columns)
        if not has_data_flattened:
            raise RuntimeError(
                "Decomposition did not flatten 'data' into parent columns. "
                "Rerun implementation cells before pipeline execution."
            )

    output_tables = [parent_table] if reassemble else [parent_table, *children.keys()]
    validate_table_names(output_tables)
    if not incremental:
        _assert_overwrite_allowed(spark, output_tables, allow_overwrite)

    if reassemble:
        flat_df = reassemble_kind(decomposed)
        if incremental:
            upsert_silver_table(flat_df, parent_table)
        else:
            write_silver_table(flat_df, parent_table, mode="overwrite")
        return KindResult(
            kind=kind,
            status="success",
            records_processed=records_processed,
            records_failed=0,
            parent_table=parent_table,
            child_tables=[],
            reassembled=True,
            validation_passed=True,
        )

    # Parent + children mode
    if incremental:
        upsert_silver_table(parent_df, parent_table)
        changed_ids_df = parent_df.select(F.col("id").cast("string").alias("id")).distinct()
        for child_table, (_, child_df) in children.items():
            _incremental_child_write(
                spark,
                child_df,
                child_table,
                changed_ids_df,
            )
    else:
        write_silver_table(parent_df, parent_table, mode="overwrite")
        for child_table, (_, child_df) in children.items():
            write_silver_table(child_df, child_table, mode="overwrite")

    return KindResult(
        kind=kind,
        status="success",
        records_processed=records_processed,
        records_failed=0,
        parent_table=parent_table,
        child_tables=list(children.keys()),
        reassembled=False,
        validation_passed=True,
    )

In [ ]:
def _check_row(name: str, passed: bool, detail: str) -> dict[str, str | bool]:
    icon = "✓" if passed else "✗"
    print(f"  {icon} {name}: {detail}")
    return {"check": name, "passed": passed, "detail": detail}


def preview_output_tables(kinds: list[str], table_prefix: str, output_mode: str) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    for kind in kinds:
        parent_table = f"{table_prefix}{kind_to_table_name(kind)}"
        validate_table_names([parent_table])
        rows.append(
            {
                "kind": kind,
                "output_mode": output_mode,
                "parent_table": parent_table,
                "child_tables": "discovered during decomposition" if output_mode == "normalized" else "not created in wide mode",
            }
        )
    return rows


def run_setup_checklist(
    spark: SparkSession,
    kinds: list[str],
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    output_mode: str,
    table_prefix: str,
) -> list[dict[str, str | bool]]:
    """Validate configuration and read access before full execution."""
    print("Setup checklist")
    checks: list[dict[str, str | bool]] = []

    checks.append(_check_row("workspace", bool(workspace_id), workspace_id or "missing"))
    checks.append(_check_row("lakehouse", bool(lakehouse_id), lakehouse_id or "missing"))
    checks.append(_check_row("bronze table", bool(bronze_table), bronze_table or "missing"))
    checks.append(_check_row("kinds", bool(kinds), f"{len(kinds)} selected" if kinds else "no kinds selected"))
    checks.append(_check_row("output mode", output_mode in {"normalized", "wide"}, output_mode))

    if kinds:
        try:
            sample_df = read_bronze_kind_spark(kinds[0], spark, workspace_id, lakehouse_id, bronze_table=bronze_table, limit=1)
            sample_rows = sample_df.take(1)
            checks.append(_check_row("bronze access", True, f"read {len(sample_rows)} preview row(s) for first kind"))
        except Exception as exc:
            checks.append(_check_row("bronze access", False, str(exc)))

        try:
            registry = build_registry_from_web([kinds[0]])
            checks.append(_check_row("schema registry", registry.has_kind(kinds[0]), f"resolved {kinds[0]}"))
        except Exception as exc:
            checks.append(_check_row("schema registry", False, str(exc)))

    planned = preview_output_tables(kinds, table_prefix, output_mode)
    metadata_tables = [globals().get("schema_cache_table", "silver_schema_cache"), globals().get("run_manifest_table", "silver_run_manifest"), "silver_run_info"]
    try:
        validate_table_names([row["parent_table"] for row in planned] + metadata_tables)
        checks.append(_check_row("table names", True, "all planned table names are valid"))
    except Exception as exc:
        checks.append(_check_row("table names", False, str(exc)))

    print("Planned output tables:")
    for row in planned:
        print(f"  - {row['parent_table']} ({row['output_mode']})")

    passed = all(bool(row["passed"]) for row in checks)
    print(f"Setup checklist {'passed' if passed else 'failed'}")
    return checks


def run_silver_dry_run(
    spark: SparkSession,
    kinds: list[str],
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str,
    limit: int | None,
    output_mode: str,
    table_prefix: str,
) -> list[dict[str, str | int | bool]]:
    """Validate inputs and preview planned writes without creating Silver Layer tables."""
    print("Dry run: validating configuration and previewing planned writes")
    checks = run_setup_checklist(spark, kinds, workspace_id, lakehouse_id, bronze_table, output_mode, table_prefix)
    dry_rows: list[dict[str, str | int | bool]] = []

    for kind in kinds:
        parent_table = f"{table_prefix}{kind_to_table_name(kind)}"
        row: dict[str, str | int | bool] = {
            "kind": kind,
            "output_mode": output_mode,
            "parent_table": parent_table,
            "schema_resolved": False,
            "preview_rows": 0,
            "would_write": False,
        }
        try:
            registry = build_registry_from_web([kind])
            row["schema_resolved"] = registry.has_kind(kind)
            preview_df = read_bronze_kind_spark(kind, spark, workspace_id, lakehouse_id, bronze_table=bronze_table, limit=limit or 1)
            row["preview_rows"] = len(preview_df.take(1))
            row["would_write"] = bool(row["schema_resolved"])
        except Exception as exc:
            row["error"] = str(exc)
        dry_rows.append(row)

    print("Dry run planned writes:")
    for row in dry_rows:
        print(f"  - {row['kind']} → {row['parent_table']} ({row['output_mode']}) schema={row['schema_resolved']} preview_rows={row['preview_rows']}")
    print("Dry run complete: no Silver Layer tables were written.")
    return dry_rows


def run_silver_build(
    spark: SparkSession,
    kinds: list[str],
    workspace_id: str,
    lakehouse_id: str,
    bronze_table: str | None = None,
    limit: int | None = None,
    incremental: bool = False,
    drop_wkt: bool = True,
    reassemble: bool = False,
    table_prefix: str = "",
    allow_overwrite: bool = False,
    run_profile: str = "full",
    notebook_version: str = "",
    config_hash: str = "",
) -> list[KindResult]:
    """Run the full silver build pipeline for one or more OSDU kinds.

    Args:
        spark: Active SparkSession.
        kinds: List of full OSDU kind strings to process.
        workspace_id: Fabric workspace GUID.
        lakehouse_id: Fabric lakehouse GUID.
        bronze_table: Name of the bronze Delta table.
        limit: Optional row limit per kind.
        incremental: If True, upsert parent rows and replace child rows for changed parent ids.
        drop_wkt: Whether to drop WKT columns.
        reassemble: If True, write one wide table per kind.
        table_prefix: Prefix for output table names (e.g. "silver_").
        allow_overwrite: If True, full refresh may replace existing output tables.
        run_profile: Effective run profile written to run metadata.
        notebook_version: Notebook version written to run metadata.
        config_hash: Short hash of effective configuration written to run metadata.

    Returns:
        List of KindResult, one per kind.
    """
    bronze_table = bronze_table or globals().get("bronze_table")
    if not bronze_table:
        raise ValueError("bronze_table is not configured")
    run_id = str(uuid.uuid4())
    results: list[KindResult] = []

    print(f"Silver build run: {run_id}")
    print(f"  Kinds: {len(kinds)}")
    print(f"  Mode: {'incremental' if incremental else 'full'}")
    output_mode = "wide" if reassemble else "normalized"
    print(f"  Output mode: {output_mode}")
    print(f"  Allow overwrite: {allow_overwrite}")
    if notebook_version:
        print(f"  Notebook version: {notebook_version}")
    if config_hash:
        print(f"  Config hash: {config_hash}")
    print(f"  Limit: {limit or 'none'}")
    print()

    for kind in kinds:
        start_time = datetime.now(UTC)
        error_message = None

        try:
            result = process_kind(
                spark,
                kind,
                workspace_id,
                lakehouse_id,
                bronze_table=bronze_table,
                limit=limit,
                incremental=incremental,
                drop_wkt=drop_wkt,
                reassemble=reassemble,
                table_prefix=table_prefix,
                allow_overwrite=allow_overwrite,
            )
            results.append(result)

        except Exception as exc:
            error_message = traceback.format_exc()
            logger.error("Failed to process %s: %s", kind, exc)
            result = KindResult(
                kind=kind,
                status="failed",
                error=str(exc),
            )
            results.append(result)

        end_time = datetime.now(UTC)

        # Write run_info
        try:
            write_run_info(
                spark,
                run_id,
                kind,
                start_time,
                end_time,
                result.records_processed,
                result.records_failed,
                result.status,
                error_message,
                workspace_id,
                lakehouse_id,
            )
        except Exception as exc:
            logger.error("Failed to write run_info for %s: %s", kind, exc)

        try:
            write_run_manifest(
                spark,
                run_id,
                result,
                output_mode,
                table_prefix,
                bronze_table,
                run_profile,
                notebook_version,
                config_hash,
                allow_overwrite,
                workspace_id,
                lakehouse_id,
            )
        except Exception as exc:
            logger.error("Failed to write run manifest for %s: %s", kind, exc)

        # Progress summary
        status_icon = {"success": "✓", "skipped": "–", "failed": "✗"}.get(result.status, "?")
        print(f"  {status_icon} {kind}")
        print(f"    table: {result.parent_table}")
        print(f"    rows:  {result.records_processed}")
        if result.reassembled:
            print("    output mode: wide")
        elif result.child_tables:
            print("    output mode: normalized")
            print(f"    children: {', '.join(result.child_tables)}")
        if result.error:
            print(f"    error: {result.error}")
        print()

    # Summary
    succeeded = sum(1 for r in results if r.status == "success")
    failed = sum(1 for r in results if r.status == "failed")
    skipped = sum(1 for r in results if r.status == "skipped")
    total_rows = sum(r.records_processed for r in results)
    print(f"Done: {succeeded} succeeded, {failed} failed, {skipped} skipped ({total_rows} total rows)")

    return results


## Setup checklist

Run this section before the smoke test or full pipeline when onboarding a new workspace. It validates the resolved tenant configuration, bronze access, schema registry access, and planned output tables without writing Silver Layer tables.


In [ ]:
# Setup checklist: validate configuration and planned outputs before writes
setup_checks = run_setup_checklist(
    spark,
    kinds=kinds,
    workspace_id=workspace_id,
    lakehouse_id=lakehouse_id,
    bronze_table=bronze_table,
    output_mode=output_mode,
    table_prefix=table_prefix,
)


## Smoke test bronze access

Run this section before full execution to validate that the configured bronze table and first selected kind can be read. The smoke test reads at most one row and does not write Silver Layer tables.


In [ ]:
# Smoke test: validate bronze access before running the full silver build
print("Smoke test starting...")
print(f"  bronze_table = {bronze_table}")
print(f"  sample kind  = {kinds[0] if kinds else '<none>'}")

if kinds:
    smoke_df = read_bronze_kind_spark(
        kinds[0],
        spark,
        workspace_id,
        lakehouse_id,
        bronze_table=bronze_table,
        limit=1,
    )
    sample_rows = smoke_df.take(1)
    print(f"  bronze preview rows = {len(sample_rows)}")
    print(f"  bronze preview cols = {len(smoke_df.columns)}")
else:
    print("  No kinds available for smoke test")


## Run pipeline

Run this section after reviewing the configuration output and completing the setup checklist and smoke test.

Profiles:

- `interactive`: print current settings and next steps without executing the pipeline.
- `dry_run`: validate selected kinds, resolve schemas, preview output tables, and avoid writes.
- `full`: process the configured kinds and write Silver Layer Delta tables.

Recommended order: configure controls, run Setup checklist, run Smoke test, set `RUN_PROFILE = "dry_run"`, run this section, then set `RUN_PROFILE = "full"` when ready.


In [ ]:
# ── Execute ───────────────────────────────────────────────────────────
if "decompose_kind" not in globals() or "reassemble_kind" not in globals() or "build_registry_from_web" not in globals():
    print("Standalone core is not loaded yet.")
    print("Run the helper and core sections first, then rerun this cell.")
    results = []
elif run_profile == "interactive":
    print("Interactive profile active: no pipeline execution was started.")
    print("Current settings:")
    print(f"  kinds selected: {len(kinds)}")
    print(f"  schema source mode: {schema_source_mode}")
    print(f"  incremental: {incremental}")
    print(f"  output mode: {output_mode}")
    print(f"  table prefix: {table_prefix}")
    print("Next steps:")
    print("  1. Update the explicit tenant configuration if needed.")
    print("  2. Run Configuration to refresh effective values.")
    print("  3. Run the setup checklist.")
    print("  4. Run the smoke test to validate bronze access.")
    print("  5. Set RUN_PROFILE to 'dry_run' for a write-free preview.")
    print("  6. Set RUN_PROFILE to 'full' when ready to write Silver Layer tables.")
    results = []
elif run_profile == "dry_run":
    dry_run_results = run_silver_dry_run(
        spark,
        kinds=kinds,
        workspace_id=workspace_id,
        lakehouse_id=lakehouse_id,
        bronze_table=bronze_table,
        limit=limit,
        output_mode=output_mode,
        table_prefix=table_prefix,
    )
    results = []
else:
    print("Full profile active: starting pipeline execution.")
    results = run_silver_build(
        spark,
        kinds=kinds,
        workspace_id=workspace_id,
        lakehouse_id=lakehouse_id,
        bronze_table=bronze_table,
        limit=limit,
        incremental=incremental,
        drop_wkt=drop_wkt,
        reassemble=reassemble,
        table_prefix=table_prefix,
        allow_overwrite=allow_overwrite,
        run_profile=run_profile,
        notebook_version=NOTEBOOK_VERSION,
        config_hash=config_hash,
    )


## Results summary

Review per-kind status, record counts, output table names, validation status, and errors after pipeline execution.


In [ ]:
# ── Results table ─────────────────────────────────────────────────────
import pandas as pd

rows = []
for r in results:
    rows.append(
        {
            "Kind": r.kind.split("--")[-1].split(":")[0],
            "Status": r.status,
            "Rows": r.records_processed,
            "Parent": r.parent_table,
            "Children": "wide" if r.reassembled else (len(r.child_tables) if r.child_tables else 0),
            "Validation": "✓" if r.validation_passed else "✗",
            "Error": r.error or "",
        }
    )

if rows:
    df_results = pd.DataFrame(rows)
    display(df_results)
elif "dry_run_results" in globals() and dry_run_results:
    display(pd.DataFrame(dry_run_results))
else:
    print("No pipeline results to display yet. Run with RUN_PROFILE = 'dry_run' or 'full'.")
